# 02 — Raw data profiling

## Objective
Inspect the source structure and data before making cleaning decisions. The source files are NASA/POWER parameter tables with 15 climate variables, yearly rows (2000–2025), JAN–DEC monthly summaries, and an `ANN` summary. Each point file has its own latitude/longitude metadata in the preamble. The documented missing sentinel is `-999`.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name.lower()=="notebooks" else Path.cwd()
sys.path.insert(0,str(ROOT/"src"))
from climate_pipeline.config import RAW_DIR, PROJECT_DIR, FINAL_DAILY_COLUMNS
from climate_pipeline import run_pipeline, validate_city_monthly, validate_daily_schema
pd.set_option("display.max_columns",60)


## Read without losing provenance

The parser searches for `-END HEADER-`, safely converts the source values, translates only documented `-999` to null, and records city, point, source filename, date, parameter, coordinates, and source. Monthly and annual summaries are kept as separate data frames.

In [2]:
from climate_pipeline.ingest import discover_files, read_all_points
inventory=discover_files(RAW_DIR)
monthly_long, annual_long=read_all_points(inventory,RAW_DIR)
print("Monthly long shape:",monthly_long.shape,"Annual long shape:",annual_long.shape)
print("Monthly date span:",monthly_long.date.min().date(),"to",monthly_long.date.max().date())
display(inventory[["source_file","city","measurement_point_id","number_of_rows","number_of_columns","date_min","date_max"]].head())
display(monthly_long.head())


Monthly long shape: (416520, 11) Annual long shape: (34710, 11)
Monthly date span: 2000-01-01 to 2025-12-01


,source_file,city,measurement_point_id,number_of_rows,number_of_columns,date_min,date_max
0,ahemdabad_1.csv,Ahmedabad,1,390,15,2000-01-01,2025-12-31
1,ahemdabad_2.csv,Ahmedabad,2,390,15,2000-01-01,2025-12-31
2,ahemdabad_3.csv,Ahmedabad,3,390,15,2000-01-01,2025-12-31
3,ahemdabad_4.csv,Ahmedabad,4,390,15,2000-01-01,2025-12-31
4,ahemdabad_5.csv,Ahmedabad,5,390,15,2000-01-01,2025-12-31


,city_key,city,measurement_point_id,source_file,date,PARAMETER,value,latitude,longitude,summary_period,source
0,ahmedabad,Ahmedabad,1,ahemdabad_1.csv,2000-01-01,ALLSKY_SFC_SW_DIFF,1.6068,22.3061,73.1826,monthly,NASA/POWER
1,ahmedabad,Ahmedabad,1,ahemdabad_1.csv,2001-01-01,ALLSKY_SFC_SW_DIFF,1.3872,22.3061,73.1826,monthly,NASA/POWER
2,ahmedabad,Ahmedabad,1,ahemdabad_1.csv,2002-01-01,ALLSKY_SFC_SW_DIFF,1.5372,22.3061,73.1826,monthly,NASA/POWER
3,ahmedabad,Ahmedabad,1,ahemdabad_1.csv,2003-01-01,ALLSKY_SFC_SW_DIFF,1.5146,22.3061,73.1826,monthly,NASA/POWER
4,ahmedabad,Ahmedabad,1,ahemdabad_1.csv,2004-01-01,ALLSKY_SFC_SW_DIFF,1.3627,22.3061,73.1826,monthly,NASA/POWER


## Profile fields and parameters

Missingness is summarized before aggregation. Numeric minima/maxima are descriptive only; extreme climate values are retained unless the source documents them as invalid. Duplicate detection keys include city, point, month, and parameter.

In [3]:
from climate_pipeline.profiling import profile_frame
summary, fields=profile_frame(monthly_long,"cleaned source monthly long table")
print(summary)
display(fields)
parameter_profile=(monthly_long.groupby("PARAMETER",as_index=False).agg(
    records=("value","size"),valid=("value","count"),missing=("value",lambda s:s.isna().sum()),
    unique_values=("value","nunique"),minimum=("value","min"),maximum=("value","max")))
parameter_profile["missing_pct"]=(100*parameter_profile.missing/parameter_profile.records).round(2)
display(parameter_profile)
print("Exact duplicates:",monthly_long.duplicated().sum())
print("Duplicate city-point-date-parameter keys:",monthly_long.duplicated(["city_key","measurement_point_id","date","PARAMETER"]).sum())


{'dataset': 'cleaned source monthly long table', 'rows': 416520, 'columns': 11, 'exact_duplicate_rows': 0}


,column,dtype,missing_count,missing_pct,unique_values
0,city_key,str,0,0.00,8
1,city,str,0,0.00,8
2,measurement_point_id,int64,0,0.00,20
3,source_file,str,0,0.00,89
4,date,datetime64[us],0,0.00,312
5,PARAMETER,string,0,0.00,15
6,value,float64,3204,0.77,17433
7,latitude,float64,0,0.00,89
8,longitude,float64,0,0.00,89
9,summary_period,str,0,0.00,1


,PARAMETER,records,valid,missing,unique_values,minimum,maximum,missing_pct
0,ALLSKY_SFC_SW_DIFF,27768,27768,0,3461,0.7579,4.7506,0.00
1,ALLSKY_SFC_SW_DNI,27768,26700,1068,4114,0.4186,7.9106,3.85
2,ALLSKY_SFC_UV_INDEX,27768,26700,1068,248,0.3200,3.0600,3.85
3,ALLSKY_SRF_ALB,27768,26700,1068,23,0.0300,0.2600,3.85
4,CLOUD_AMT,27768,27768,0,3599,0.2900,98.6600,0.00
5,CLRSKY_SFC_SW_DWN,27768,27768,0,3830,2.8073,8.0599,0.00
6,IMERG_PRECTOT,27768,27768,0,1369,0.0000,29.6400,0.00
7,PRECTOTCORR,27768,27768,0,1008,0.0000,33.5900,0.00
8,PS,27768,27768,0,790,90.0900,101.1100,0.00
9,QV2M,27768,27768,0,1609,2.3400,21.3900,0.00


Exact duplicates: 0
Duplicate city-point-date-parameter keys: 0


## Source interpretation

The monthly files cannot provide a daily date series. They do, however, support city-month analysis. Annual values remain available in a separate long table and are not mixed with monthly values.